# Week 2 Mini-Assignment: Texas last statements by Alissa Rivero

**Dataset:** [Last Statements of Executed Offenders](https://www.kaggle.com/datasets/ranjithkumarraik/last-words-of-death-row-inmates) (TDCJ / Kaggle)

- **PreviousCrime = 0** (n = 233) = no prior criminal record
- **PreviousCrime = 1** (n = 276) = prior criminal record
- **PreviousCrime missing** (n = 36) = dropped from the models

This notebook is the same analysis in **Rust**. Pick the **Rust** kernel (Select Another Kernel → Jupyter Kernel → Rust). If `let` is a `SyntaxError`, you are still on Python. Ownership is handled with `.clone()`, `&` borrow, and dropping readers before a write.

## Dataset Abstract

The Texas Department of Criminal Justice publishes the last statements of people executed in Texas, together with age, race, county of conviction, education, prior-crime history, and victim counts. This table is a Kaggle export of that archive (545 rows). Statements are short and uneven: some people speak at length, some give a sentence, and about one in five have no recorded statement.

Theme scores are **rates per 100 words**. Keyword lists are exact words or clear prefixes. Neutral verbs such as *ask* and *tell* are not treated as themes.

## Research Questions

1. Is a **prior criminal record** associated with more or less **apology / remorse** language?
2. Is a **prior criminal record** associated with more or less **religious** language?
3. What **themes** appear in last words — remorse, gratitude and love, family, and religion — and are those rates predicted by **demographic categories**?


## Load and inspect the dataset


In [ ]:
:dep csv = "1.3"

In [ ]:
use std::collections::HashMap;

fn tokenize(text: &str) -> Vec<String> {
    let mut words = Vec::new();
    let mut current = String::new();
    for ch in text.to_lowercase().chars() {
        if ch.is_ascii_alphabetic() || ch == '\'' {
            current.push(ch);
        } else if !current.is_empty() {
            words.push(std::mem::take(&mut current));
        }
    }
    if !current.is_empty() {
        words.push(current);
    }
    words
}

fn is_declined(text: &str) -> bool {
    let low = text.trim().to_ascii_lowercase();
    low.is_empty() || low == "none" || low.contains("declined") || low.contains("no last statement")
}

fn is_hit(word: &str, exact: &[&str], prefixes: &[&str]) -> bool {
    exact.iter().any(|w| *w == word) || prefixes.iter().any(|p| word.starts_with(p))
}

fn theme_hits(words: &[String], exact: &[&str], prefixes: &[&str]) -> f64 {
    words.iter().filter(|w| is_hit(w, exact, prefixes)).count() as f64
}

const REMORSE_EXACT: &[&str] = &[
    "sorry", "sorrow", "apology", "apologize", "apologise", "apologized",
    "apologised", "apologies", "forgive", "forgave", "forgiven", "forgiveness",
    "forgiving", "remorse", "remorseful", "regret", "regrets", "regretted",
    "regretting", "repent", "repents", "repented", "repentance",
];
const REMORSE_PREFIXES: &[&str] = &["apolog", "forgiv", "remorse", "regret", "repent"];
const GRAT_EXACT: &[&str] = &[
    "thank", "thanks", "thankful", "thankfully", "gratitude", "grateful",
    "appreciate", "appreciated", "appreciation", "love", "loved", "loves",
    "loving", "lovin",
];
const GRAT_PREFIXES: &[&str] = &["thank", "gratitude", "grateful", "appreciate"];
const FAMILY_EXACT: &[&str] = &[
    "family", "families", "mom", "moms", "mommy", "mama", "mother", "mothers",
    "mum", "dad", "dads", "daddy", "father", "fathers", "papa", "parent",
    "parents", "wife", "wives", "husband", "husbands", "son", "sons",
    "daughter", "daughters", "kid", "kids", "child", "children", "brother",
    "brothers", "bro", "sister", "sisters", "sis", "sibling", "siblings",
    "grandma", "grandmother", "grandpa", "grandfather", "grandparent",
    "grandparents", "aunt", "aunts", "uncle", "uncles", "nephew", "nephews",
    "niece", "nieces", "cousin", "cousins", "friend", "friends",
    "friendship", "baby", "babies",
];
const FAMILY_PREFIXES: &[&str] = &[
    "mother", "father", "daughter", "brother", "sister", "grandma",
    "grandpa", "grandparent", "friend",
];
const REL_EXACT: &[&str] = &[
    "god", "gods", "godly", "jesus", "christ", "christian", "christianity",
    "lord", "lords", "heaven", "heavenly", "allah", "pray", "prayer",
    "prayers", "praying", "prayed", "bible", "biblical", "amen", "holy",
    "church", "faith", "bless", "blessed", "blessing", "blessings",
];
const REL_PREFIXES: &[&str] = &["jesus", "christ", "heaven", "pray", "prayer", "bible", "bless"];

#[derive(Clone)]
struct Person {
    last_name: String,
    race: String,
    age: Option<f64>,
    education: Option<f64>,
    prior: Option<f64>,
    statement: String,
    declined: bool,
    word_count: f64,
    apology_rate: f64,
    religion_rate: f64,
    gratitude_love_rate: f64,
    family_rate: f64,
}

fn rate(hits: f64, words: f64) -> f64 {
    if words <= 0.0 { 0.0 } else { hits / words * 100.0 }
}

fn load_people(path: &str) -> Vec<Person> {
    let bytes = std::fs::read(path).expect("read csv");
    let text: String = bytes.iter().copied().map(char::from).collect(); // Latin-1
    let mut rdr = csv::ReaderBuilder::new().flexible(true).from_reader(text.as_bytes());
    let headers = rdr.headers().expect("headers").clone();
    let idx = |name: &str| {
        headers.iter().position(|h| h.trim() == name).unwrap_or_else(|| panic!("missing {name}"))
    };
    let i_last = idx("LastName");
    let i_race = idx("Race");
    let i_age = idx("Age");
    let i_edu = idx("EducationLevel");
    let i_prior = idx("PreviousCrime");
    let i_stmt = idx("LastStatement");
    let parse_f = |s: &str| {
        let t = s.trim();
        if t.is_empty() || t.eq_ignore_ascii_case("NA") { None } else { t.parse::<f64>().ok() }
    };
    let mut people = Vec::new();
    for rec in rdr.records() {
        let rec = rec.expect("row");
        let statement = rec.get(i_stmt).unwrap_or("").to_string();
        let declined = is_declined(&statement);
        let words = if declined { Vec::new() } else { tokenize(&statement) };
        let n = words.len() as f64;
        people.push(Person {
            last_name: rec.get(i_last).unwrap_or("").to_string(),
            race: rec.get(i_race).unwrap_or("").to_string(),
            age: parse_f(rec.get(i_age).unwrap_or("")),
            education: parse_f(rec.get(i_edu).unwrap_or("")),
            prior: parse_f(rec.get(i_prior).unwrap_or("")),
            statement,
            declined,
            word_count: n,
            apology_rate: rate(theme_hits(&words, REMORSE_EXACT, REMORSE_PREFIXES), n),
            religion_rate: rate(theme_hits(&words, REL_EXACT, REL_PREFIXES), n),
            gratitude_love_rate: rate(theme_hits(&words, GRAT_EXACT, GRAT_PREFIXES), n),
            family_rate: rate(theme_hits(&words, FAMILY_EXACT, FAMILY_PREFIXES), n),
        });
    }
    people
}

let people = load_people("Texas Last Statement - CSV.csv");
println!("rows: {}", people.len());
println!("columns after strip: LastName Race Age EducationLevel PreviousCrime LastStatement + scores");
println!("first 5 last names:");
for p in people.iter().take(5) {
    println!("  {}  race={}  prior={:?}  words={:.0}", p.last_name, p.race, p.prior, p.word_count);
}


In [ ]:
fn mean(xs: &[f64]) -> f64 { xs.iter().sum::<f64>() / xs.len() as f64 }
fn std_sample(xs: &[f64]) -> f64 {
    if xs.len() < 2 { return f64::NAN; }
    let m = mean(xs);
    let v = xs.iter().map(|x| (x - m).powi(2)).sum::<f64>() / (xs.len() as f64 - 1.0);
    v.sqrt()
}

let ages: Vec<f64> = people.iter().filter_map(|p| p.age).collect();
let words: Vec<f64> = people.iter().map(|p| p.word_count).collect();
println!("Age    min={:.0} max={:.0} mean={:.1}", ages.iter().cloned().fold(f64::INFINITY, f64::min), ages.iter().cloned().fold(f64::NEG_INFINITY, f64::max), mean(&ages));
println!("words  min={:.0} max={:.0} mean={:.1}", words.iter().cloned().fold(f64::INFINITY, f64::min), words.iter().cloned().fold(f64::NEG_INFINITY, f64::max), mean(&words));
let missing_prior = people.iter().filter(|p| p.prior.is_none()).count();
let declined = people.iter().filter(|p| p.declined).count();
println!("missing PreviousCrime: {missing_prior}");
println!("declined / no statement: {declined}");


## Split groups


In [ ]:
let no_prior: Vec<&Person> = people.iter().filter(|p| p.prior == Some(0.0)).collect();
let prior: Vec<&Person> = people.iter().filter(|p| p.prior == Some(1.0)).collect();
println!("No prior record: {}", no_prior.len());
println!("Prior record: {}", prior.len());
println!("first no-prior: {}", no_prior[0].last_name);
println!("first prior: {}", prior[0].last_name);


## Ownership: clone, then move

`.clone()` makes a second list so both names can live after the move.


In [ ]:
let names = vec![
    people[0].last_name.clone(),
    people[1].last_name.clone(),
    people[2].last_name.clone(),
];
let copy = names.clone(); // second list, so both names can live
let moved = names;        // move is fine: we still have copy
println!("moved = {moved:?}");
println!("copy  = {copy:?}");


`moved` owns the original list. `copy` owns the clone.


In [ ]:
fn count_owned(words: Vec<String>) -> usize { words.len() }
fn count_borrowed(words: &[String]) -> usize { words.len() }

let words = vec![String::from("sorry"), String::from("family")];
println!("borrowed count = {}; still own {:?}", count_borrowed(&words), words);

let n = count_owned(words.clone()); // clone so the caller still owns words
println!("owned-count on a clone = {n}; still own {words:?}");


`count_borrowed(&words)` looks without taking. A clone is passed into `count_owned` so the caller still owns `words`.


## `groupby()` summary statistics

Group by prior-crime status, then by race.


In [ ]:
fn print_group(label: &str, rows: &[&Person]) {
    let words: Vec<f64> = rows.iter().map(|p| p.word_count).collect();
    let ages: Vec<f64> = rows.iter().filter_map(|p| p.age).collect();
    let apology: Vec<f64> = rows.iter().map(|p| p.apology_rate).collect();
    let religion: Vec<f64> = rows.iter().map(|p| p.religion_rate).collect();
    let grat: Vec<f64> = rows.iter().map(|p| p.gratitude_love_rate).collect();
    let family: Vec<f64> = rows.iter().map(|p| p.family_rate).collect();
    println!("{label:10} n={:<4} words={:.1} age={:.1} apology={:.2} religion={:.2} gratitude={:.2} family={:.2}",
        rows.len(), mean(&words), mean(&ages), mean(&apology), mean(&religion), mean(&grat), mean(&family));
}

println!("group      n    words  age   apology religion gratitude family");
print_group("No prior", &no_prior);
print_group("Prior", &prior);

println!();
println!("by Race");
let mut by_race: HashMap<String, Vec<&Person>> = HashMap::new();
for p in &people {
    by_race.entry(p.race.clone()).or_default().push(p);
}
let mut races: Vec<_> = by_race.keys().cloned().collect();
races.sort();
for race in races {
    print_group(&race, &by_race[&race]);
}


## Side-by-side comparison


In [ ]:
println!("Race       metric              No prior   Prior    percent_diff");
for race in ["White", "Black", "Hispanic"] {
    let a: Vec<&Person> = no_prior.iter().copied().filter(|p| p.race == race).collect();
    let b: Vec<&Person> = prior.iter().copied().filter(|p| p.race == race).collect();
    let metrics = [
        ("word_count", a.iter().map(|p| p.word_count).sum::<f64>() / a.len() as f64,
                       b.iter().map(|p| p.word_count).sum::<f64>() / b.len() as f64),
        ("apology_rate", mean(&a.iter().map(|p| p.apology_rate).collect::<Vec<_>>()),
                         mean(&b.iter().map(|p| p.apology_rate).collect::<Vec<_>>())),
        ("religion_rate", mean(&a.iter().map(|p| p.religion_rate).collect::<Vec<_>>()),
                          mean(&b.iter().map(|p| p.religion_rate).collect::<Vec<_>>())),
        ("gratitude_love", mean(&a.iter().map(|p| p.gratitude_love_rate).collect::<Vec<_>>()),
                           mean(&b.iter().map(|p| p.gratitude_love_rate).collect::<Vec<_>>())),
        ("family_rate", mean(&a.iter().map(|p| p.family_rate).collect::<Vec<_>>()),
                        mean(&b.iter().map(|p| p.family_rate).collect::<Vec<_>>())),
    ];
    for (name, na, pr) in metrics {
        let pct = if na == 0.0 { f64::NAN } else { (pr - na) / na * 100.0 };
        println!("{race:10} {name:18} {na:8.3} {pr:8.3} {pct:10.2}");
    }
}


## Composite theme scores

Four last-word themes, counted as **hits per 100 words**:

- **Remorse** (apology) — *sorry, apologize, forgive, remorse, regret*
- **Gratitude and love** (one theme) — *thank, grateful, love*
- **Family** (kept separate) — *mom, kids, brother, wife, family, friends*
- **Religion** — *god, jesus, christ, lord, heaven, pray, amen*

Neutral words such as *ask* and *tell* are not included.


In [ ]:
println!("score              group     mean    count    std");
let pairs = [
    ("apology_rate", no_prior.iter().map(|p| p.apology_rate).collect::<Vec<_>>(), prior.iter().map(|p| p.apology_rate).collect::<Vec<_>>()),
    ("religion_rate", no_prior.iter().map(|p| p.religion_rate).collect::<Vec<_>>(), prior.iter().map(|p| p.religion_rate).collect::<Vec<_>>()),
    ("gratitude_love", no_prior.iter().map(|p| p.gratitude_love_rate).collect::<Vec<_>>(), prior.iter().map(|p| p.gratitude_love_rate).collect::<Vec<_>>()),
    ("family_rate", no_prior.iter().map(|p| p.family_rate).collect::<Vec<_>>(), prior.iter().map(|p| p.family_rate).collect::<Vec<_>>()),
];
for (name, a, b) in &pairs {
    println!("{name:18} No prior  {:.4}   {}       {:.4}", mean(a), a.len(), std_sample(a));
    println!("{name:18} Prior     {:.4}   {}       {:.4}", mean(b), b.len(), std_sample(b));
}


### Ownership: many readers, then one writer

Two `&` loans at once, then they drop, then a write.


In [ ]:
let mut themes = vec![String::from("remorse"), String::from("family")];
{
    let first = &themes;
    let second = &themes;
    println!("two readers: {first:?} and {second:?}");
} // both loans end here

themes.push(String::from("religion"));
println!("after exclusive write: {themes:?}");


A function can take `&mut Vec<String>` for the same exclusive write.


In [ ]:
fn add_theme(themes: &mut Vec<String>, name: String) {
    themes.push(name); // exclusive write through a mutable loan
}

let mut themes = vec![String::from("remorse"), String::from("family")];
add_theme(&mut themes, String::from("religion"));
println!("after &mut write: {themes:?}");


## Prediction models

`apology_rate ~ prior_crime` and `religion_rate ~ prior_crime` (prior_crime = 1 for a prior record, 0 for none). Then each theme rate ~ prior_crime + Age + EducationLevel + Race.


In [ ]:
/// OLS of y ~ 1 + x. Returns (intercept, slope, r2, se_slope, t_slope).
fn ols(y: &[f64], x: &[f64]) -> (f64, f64, f64, f64, f64) {
    let n = y.len() as f64;
    let my = mean(y);
    let mx = mean(x);
    let sxx: f64 = x.iter().map(|v| (v - mx).powi(2)).sum();
    let sxy: f64 = x.iter().zip(y).map(|(a, b)| (a - mx) * (b - my)).sum();
    let syy: f64 = y.iter().map(|v| (v - my).powi(2)).sum();
    let slope = sxy / sxx;
    let intercept = my - slope * mx;
    let sse: f64 = y.iter().zip(x).map(|(yi, xi)| {
        let pred = intercept + slope * xi;
        (yi - pred).powi(2)
    }).sum();
    let r2 = 1.0 - sse / syy;
    let se_slope = (sse / (n - 2.0) / sxx).sqrt();
    let t_slope = slope / se_slope;
    (intercept, slope, r2, se_slope, t_slope)
}

fn erf(z: f64) -> f64 {
    let x = z.abs();
    let t = 1.0 / (1.0 + 0.3275911 * x);
    let y = 1.0 - t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429)))) * (-x * x).exp();
    if z >= 0.0 { y } else { -y }
}
fn two_sided_p_norm(t: f64) -> f64 {
    2.0 * (1.0 - (1.0 + erf(t.abs() / std::f64::consts::SQRT_2)) / 2.0)
}

fn solve(mut a: Vec<Vec<f64>>, mut b: Vec<f64>) -> Vec<f64> {
    let k = b.len();
    for i in 0..k {
        let mut piv = i;
        for r in (i + 1)..k {
            if a[r][i].abs() > a[piv][i].abs() { piv = r; }
        }
        a.swap(i, piv);
        b.swap(i, piv);
        let diag = a[i][i];
        for c in i..k { a[i][c] /= diag; }
        b[i] /= diag;
        for r in 0..k {
            if r == i { continue; }
            let f = a[r][i];
            for c in i..k { a[r][c] -= f * a[i][c]; }
            b[r] -= f * b[i];
        }
    }
    b
}

fn ols_multi(y: &[f64], x: &[Vec<f64>]) -> (Vec<f64>, f64) {
    let n = y.len();
    let k = x[0].len();
    let mut xtx = vec![vec![0.0; k]; k];
    let mut xty = vec![0.0; k];
    for i in 0..n {
        for a in 0..k {
            xty[a] += x[i][a] * y[i];
            for b in 0..k {
                xtx[a][b] += x[i][a] * x[i][b];
            }
        }
    }
    let beta = solve(xtx, xty);
    let my = mean(y);
    let sse: f64 = y.iter().enumerate().map(|(i, yi)| {
        let pred: f64 = beta.iter().zip(&x[i]).map(|(b, v)| b * v).sum();
        (yi - pred).powi(2)
    }).sum();
    let sst: f64 = y.iter().map(|yi| (yi - my).powi(2)).sum();
    (beta, 1.0 - sse / sst)
}

let labeled: Vec<&Person> = people.iter().filter(|p| p.prior.is_some()).collect();
let y_ap: Vec<f64> = labeled.iter().map(|p| p.apology_rate).collect();
let y_rel: Vec<f64> = labeled.iter().map(|p| p.religion_rate).collect();
let x_prior: Vec<f64> = labeled.iter().map(|p| p.prior.unwrap()).collect();
let (a0, a1, ar2, ase, at) = ols(&y_ap, &x_prior);
let (r0, r1, rr2, rse, rt) = ols(&y_rel, &x_prior);

println!("apology_rate ~ prior_crime");
println!("  intercept (no-prior mean) = {a0:.4}");
println!("  prior_crime               = {a1:.4}   se = {ase:.4}   t = {at:.3}   p ≈ {:.4}", two_sided_p_norm(at));
println!("  R²                        = {ar2:.3}");
println!();
println!("religion_rate ~ prior_crime");
println!("  intercept (no-prior mean) = {r0:.4}");
println!("  prior_crime               = {r1:.4}   se = {rse:.4}   t = {rt:.3}   p ≈ {:.4}", two_sided_p_norm(rt));
println!("  R²                        = {rr2:.3}");


In [ ]:
let demo: Vec<&Person> = people.iter().filter(|p| {
    p.prior.is_some() && p.age.is_some() && p.education.is_some()
        && (p.race == "White" || p.race == "Black" || p.race == "Hispanic")
}).collect();

let x_demo: Vec<Vec<f64>> = demo.iter().map(|p| {
    vec![
        1.0,
        p.prior.unwrap(),
        p.age.unwrap(),
        p.education.unwrap(),
        if p.race == "Black" { 1.0 } else { 0.0 },
        if p.race == "Hispanic" { 1.0 } else { 0.0 },
    ]
}).collect();

let outcomes = [
    ("remorse_rate", demo.iter().map(|p| p.apology_rate).collect::<Vec<_>>()),
    ("gratitude_love_rate", demo.iter().map(|p| p.gratitude_love_rate).collect::<Vec<_>>()),
    ("family_rate", demo.iter().map(|p| p.family_rate).collect::<Vec<_>>()),
    ("religion_rate", demo.iter().map(|p| p.religion_rate).collect::<Vec<_>>()),
];
let names = ["const", "prior_crime", "Age", "EducationLevel", "Race_Black", "Race_Hispanic"];
for (label, y) in &outcomes {
    let (beta, r2) = ols_multi(y, &x_demo);
    println!("{label} ~ prior_crime + Age + EducationLevel + Race   n={}  R²={:.3}", y.len(), r2);
    for (name, b) in names.iter().zip(&beta) {
        println!("  {name:16} {b:8.4}");
    }
    println!();
}


### Interpretation: apology / remorse by prior crime

**Question 1: Is a prior criminal record associated with more or less apology / remorse language?**
**No clear difference** in this sample.

- **const ≈ 1.07:** people with no prior record use about 1.07 remorse words per 100 words.
- **prior_crime ≈ 0:** people with a prior record are essentially the same. The interval crosses zero.
- **p ≈ 0.99** and **R² ≈ 0:** prior crime does not predict the remorse score.


### Interpretation: religion by prior crime

**Question 2: Is a prior criminal record associated with more or less religious language?**
A **small increase**, not significant at 0.05.

- **const ≈ 1.45:** no-prior statements average about 1.45 religion words per 100 words.
- **prior_crime ≈ +0.50:** prior-record statements are about half a word per 100 higher. The interval crosses zero.
- **p ≈ 0.07** and **R² = 0.006:** only suggestive.


### Interpretation: themes and demographics

The four themes that appear are remorse, gratitude/love, family, and religion. *Ask* and *tell* are not themes.

- **Remorse:** Black speakers use fewer remorse words per 100 than White speakers. Prior crime, age, and education are not associated with remorse.
- **Gratitude and love:** Hispanic speakers use more gratitude/love language than White speakers. A prior record is only a borderline increase. This is the strongest demographic model.
- **Family:** common in every group and not predicted by race, age, education, or prior crime.
- **Religion:** Hispanic speakers use more religious language than White speakers. Prior crime is again only suggestive.

**Takeaway:** last-word themes are mostly gratitude/love and family. Demographics explain only a small share of the rates. A prior record does not change remorse.


## Visualization

A **boxplot plus strip plot** is the right chart here: two groups and a continuous rate. The box shows the distribution. The points keep every person visible. Each panel has its own y-axis because the two rates have different typical ranges.


In [ ]:
fn quartiles(mut xs: Vec<f64>) -> (f64, f64, f64, f64, f64) {
    xs.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let q = |p: f64| {
        let i = (p * (xs.len() as f64 - 1.0)).floor() as usize;
        xs[i]
    };
    (xs[0], q(0.25), q(0.5), q(0.75), *xs.last().unwrap())
}

fn jitter(i: usize) -> f64 {
    let u = ((i.wrapping_mul(1103515245) + 12345) % 1000) as f64 / 1000.0;
    (u - 0.5) * 28.0
}

fn panel_svg(title: &str, left_vals: &[f64], right_vals: &[f64], fill_l: &str, fill_r: &str, edge: &str, y_max: f64, x0: f64, left_name: &str, right_name: &str) -> String {
    let w = 280.0;
    let h = 280.0;
    let left = 48.0;
    let bottom = 36.0;
    let top = 28.0;
    let plot_h = h - top - bottom;
    let yx = |v: f64| top + plot_h * (1.0 - v / y_max);
    let mut s = String::new();
    s.push_str(&format!("<g transform='translate({x0},0)'>"));
    s.push_str(&format!("<text x='{:.1}' y='18' text-anchor='middle' font-size='14' font-weight='600' fill='{edge}'>{title}</text>", left + (w-left)/2.0));
    s.push_str(&format!("<line x1='{left}' y1='{top}' x2='{left}' y2='{:.1}' stroke='#C9C3B8'/>", h-bottom));
    s.push_str(&format!("<line x1='{left}' y1='{:.1}' x2='{w}' y2='{:.1}' stroke='#C9C3B8'/>", h-bottom, h-bottom));
    for tick in 0..5 {
        let v = y_max * tick as f64 / 4.0;
        let y = yx(v);
        s.push_str(&format!("<line x1='{left}' y1='{y:.1}' x2='{w}' y2='{y:.1}' stroke='#E6E1D8'/>"));
        s.push_str(&format!("<text x='44' y='{:.1}' text-anchor='end' font-size='10' fill='#5A554E'>{:.1}</text>", y + 3.0, v));
    }
    let groups = [(left_name, left_vals, fill_l, 110.0), (right_name, right_vals, fill_r, 200.0)];
    for (name, vals, fill, cx) in groups {
        let (_mn, q1, med, q3, mx) = quartiles(vals.to_vec());
        let box_w = 36.0;
        s.push_str(&format!(
            "<rect x='{:.1}' y='{:.1}' width='{box_w}' height='{:.1}' fill='{fill}' stroke='{edge}' stroke-width='1.2' rx='2'/>",
            cx - box_w / 2.0, yx(q3), (yx(q1) - yx(q3)).abs().max(1.0)
        ));
        s.push_str(&format!("<line x1='{:.1}' y1='{:.1}' x2='{:.1}' y2='{:.1}' stroke='{edge}' stroke-width='1.2'/>", cx - 12.0, yx(med), cx + 12.0, yx(med)));
        s.push_str(&format!("<line x1='{cx}' y1='{:.1}' x2='{cx}' y2='{:.1}' stroke='{edge}'/>", yx(q3), yx(mx)));
        s.push_str(&format!("<line x1='{cx}' y1='{:.1}' x2='{cx}' y2='{:.1}' stroke='{edge}'/>", yx(q1), yx(_mn)));
        for (i, v) in vals.iter().enumerate() {
            let x = cx + jitter(i);
            let y = yx(*v);
            s.push_str(&format!("<circle cx='{x:.1}' cy='{y:.1}' r='2.4' fill='{fill}' stroke='white' stroke-width='0.8' opacity='0.55'/>"));
        }
        s.push_str(&format!("<text x='{cx}' y='{:.1}' text-anchor='middle' font-size='12' fill='#2F2B26'>{name}</text>", h - 12.0));
    }
    s.push_str(&format!("<text x='16' y='{:.1}' transform='rotate(-90 16 {:.1})' font-size='11' fill='#5A554E'>Hits per 100 words</text>", h/2.0, h/2.0));
    s.push_str("</g>");
    s
}

let ap_no: Vec<f64> = no_prior.iter().map(|p| p.apology_rate).collect();
let ap_yes: Vec<f64> = prior.iter().map(|p| p.apology_rate).collect();
let rel_no: Vec<f64> = no_prior.iter().map(|p| p.religion_rate).collect();
let rel_yes: Vec<f64> = prior.iter().map(|p| p.religion_rate).collect();
let ap_max = ap_no.iter().chain(&ap_yes).cloned().fold(0.0, f64::max) * 1.15;
let rel_max = rel_no.iter().chain(&rel_yes).cloned().fold(0.0, f64::max) * 1.15;
let svg = format!(
    "<svg xmlns='http://www.w3.org/2000/svg' width='640' height='320' style='background:#FBF9F6'>\
     <text x='320' y='22' text-anchor='middle' font-size='16' font-weight='600' fill='#2F2B26'>Last-statement language by prior criminal record</text>\
     {}{}</svg>",
    panel_svg("Apology / remorse", &ap_no, &ap_yes, "#D7E2B4", "#5E7A32", "#3F5320", ap_max, 10.0, "No prior", "Prior"),
    panel_svg("Religious language", &rel_no, &rel_yes, "#F3D6E0", "#B56B86", "#7A3F56", rel_max, 330.0, "No prior", "Prior"),
);
println!("EVCXR_BEGIN_CONTENT image/svg+xml\n{svg}\nEVCXR_END_CONTENT");
